# Data Merging
**Requirements:**
- Place the *.ipynb* file and the datasets folder in the same directory.
- `Dataset/` -> Raw Data. Unzip the datasets mentioned in *Input Sources* to this folder.
- `Merged_Dataset/` -> Merged Dataset

**Input Sources:**
- `icu/icustays`: icu admissions, discharge, los. etc.
- `hosp/admissions`: hospital-wide admission and discharge records for all patients
- `hosp/patients`: demographic and background information of patients
- `icu/d_items`: metadata describing events corresponding to each `item_id`
- `icu/chart_events`: charted items occurring during the ICU stay

**Output:**
- `icu_patients`: admission and discharge information for all ICU patients, with anchor years randomly assigned.
- `dynamics`: daily inflow and outflow of each ICU care unit, aggregated and summarized using pivot tables.
- `chart_events`: selected clinical events documented during patients’ ICU stays

In [1]:
import pandas as pd
import random
import os
from tqdm import tqdm

random.seed(5188)

input_folder = 'Dataset/'
output_folder = 'Merged_Dataset/'
os.makedirs(os.path.dirname(output_folder), exist_ok=True)

## ICU Patients Profiles

In [2]:
# load the data
admissions = pd.read_csv(input_folder + 'admissions.csv')
icustays = pd.read_csv(input_folder + 'icustays.csv')
patients = pd.read_csv(input_folder + 'patients.csv')

# merge the datasets
icu_patients = pd.merge(icustays, patients)
icu_patients = pd.merge(icu_patients, admissions)

# remove unnecessary columns and change datetime types
icu_patients.drop(columns=["admit_provider_id", "edregtime", "edouttime", "dod", "deathtime", "admittime","dischtime", "last_careunit"], inplace=True)
icu_patients[["intime", "outtime"]] = icu_patients[["intime", "outtime"]].apply(pd.to_datetime, errors="coerce")

Categorize care unit types for further analysis.
- Standardize care unit names, as some represent the same unit but are recorded under different variations.
- Categories can be adjusted or refined later if necessary.

In [3]:
careunit_map = {
    # medical ICU
    'Medical Intensive Care Unit (MICU)': 'MICU',
    'Medicine': 'MICU',
    # surgical ICU
    'Surgical Intensive Care Unit (SICU)': 'SICU',
    'Medical/Surgical Intensive Care Unit (MICU/SICU)': 'MICU/SICU',
    # medical + surgical
    'Med/Surg': 'MICU/SICU',
    # cardiac ICU
    'Cardiac Vascular Intensive Care Unit (CVICU)': 'Cardiac ICU',
    'Coronary Care Unit (CCU)': 'Cardiac ICU',
    # trauma+surgery
    'Trauma SICU (TSICU)': 'TSICU',
    'Surgery/Trauma': 'SICU',
    # neuro
    'Neuro Surgical Intensive Care Unit (Neuro SICU)': 'Neuro-ICU',
    'Neurology': 'Neuro-ICU',
    'Neuro Intermediate': 'Neuro-ICU',
    'Neuro Stepdown': 'Neuro-ICU',
    # others
    'Intensive Care Unit (ICU)': 'General-ICU',
    'PACU': 'PACU',
    # intermediate
    'Medicine/Cardiology Intermediate': 'Intermediate',
    'Surgery/Vascular/Intermediate': 'Intermediate',
}
icu_patients["icu_types"] = icu_patients["first_careunit"].apply(lambda x: careunit_map[x])

icu_patients

,subject_id,hadm_id,stay_id,first_careunit,intime,outtime,los,gender,anchor_age,anchor_year,anchor_year_group,admission_type,admission_location,discharge_location,insurance,language,marital_status,race,hospital_expire_flag,icu_types
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266,F,52,2180,2014 - 2016,EW EMER.,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,0,MICU
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252,F,86,2150,2008 - 2010,EW EMER.,EMERGENCY ROOM,REHAB,Medicare,English,WIDOWED,WHITE,0,MICU
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535,F,73,2186,2008 - 2010,EW EMER.,EMERGENCY ROOM,HOME HEALTH CARE,Medicare,English,MARRIED,BLACK/AFRICAN AMERICAN,0,MICU
3,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032,F,55,2157,2011 - 2013,EW EMER.,EMERGENCY ROOM,HOME HEALTH CARE,Private,Other,MARRIED,WHITE,0,SICU
4,10001217,27703517,34592300,Surgical Intensive Care Unit (SICU),2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113,F,55,2157,2011 - 2013,DIRECT EMER.,PHYSICIAN REFERRAL,HOME HEALTH CARE,Private,Other,MARRIED,WHITE,0,SICU
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94453,19999442,26785317,32336619,Surgical Intensive Care Unit (SICU),2148-11-19 14:23:43,2148-11-26 13:12:15,6.950370,M,41,2146,2008 - 2010,ELECTIVE,PHYSICIAN REFERRAL,REHAB,Medicaid,English,DIVORCED,WHITE,0,SICU
94454,19999625,25304202,31070865,Medical/Surgical Intensive Care Unit (MICU/SICU),2139-10-10 19:18:00,2139-10-11 18:21:28,0.960741,M,81,2138,2008 - 2010,EW EMER.,EMERGENCY ROOM,HOME HEALTH CARE,Medicare,Modern Greek (1453-),MARRIED,WHITE,0,MICU/SICU
94455,19999828,25744818,36075953,Medical Intensive Care Unit (MICU),2149-01-08 18:12:00,2149-01-10 13:11:02,1.790995,F,46,2147,2017 - 2019,EW EMER.,TRANSFER FROM HOSPITAL,HOME HEALTH CARE,Medicaid,English,SINGLE,WHITE,0,MICU
94456,19999840,21033226,38978960,Surgical Intensive Care Unit (SICU),2164-09-12 09:26:28,2164-09-17 16:35:15,5.297766,M,58,2164,2008 - 2010,EW EMER.,EMERGENCY ROOM,DIED,Private,English,WIDOWED,WHITE,1,SICU


**Random assignment of admission year**, based on the `anchor_year` and `anchor_year_group`.
-  `anchor_year`: a shifted year that masks the patient’s true admission year
-  `anchor_year_group`: a range of actual calendar years during which the admission could have occurred
- e.g. If `anchor_year=2180` and `anchor_year_group="2014–2016"`, the actual admission took place sometime between 2014 and 2016.

However, there is no one-to-one mapping between anchor_year and the true calendar year, so we cannot directly infer or aggregate by `anchor_year`. Therefore, I choose to Rrndomly assign the admission year from within the `anchor_year_group` while keeping LoS and admission-discharge date.
- For leap day, if the `anchor_year_group` includes a leap year, assign it when possible. Otherwise, convert February 29 to February 28.

In [ ]:
def is_leap_year(year):
    return (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0)

def randomly_assign_year(intime, outtime, anchor_year, anchor_year_group):
    """
    Randomly assign the year within anchor_year_group, keeping LOS and handling leap day.
    """
    # anchor_year_group
    anchor_start = int(anchor_year_group.split("-")[0]) + intime.year - anchor_year
    yr_range = list(range(anchor_start, anchor_start + 3))

    # manage leap year, i.e. 29/2
    # no need to consider outtime as it's generated by intime+inout_delta
    if intime.month == 2 and intime.day == 29:
        yr_range = [y for y in yr_range if is_leap_year(y)]
        if not yr_range:
            # fallback: map 29/2 -> 28/2
            intime = intime.replace(day=28)
            yr_range = list(range(anchor_start, anchor_start + 3))
    new_year = random.choice(yr_range)
    inout_delta = outtime - intime
    new_intime = intime.replace(year=new_year)
    new_outtime = new_intime + inout_delta

    return new_intime, new_outtime

In [77]:
icu_patients_copy = icu_patients.copy()

# apply function row-wise
icu_patients[["assigned_intime", "assigned_outtime"]] = icu_patients.apply(
    lambda row: randomly_assign_year(
        row["intime"], row["outtime"], row["anchor_year"], row["anchor_year_group"]
    ),
    axis=1, result_type="expand"
)

### Output Description
- `subject_id`: unique identifier for each patient
- `stay_id`: unique identifier for each ICU stay (key)
- `hadm_id`: unique identifier for each hospital admission
- `intime / outtime`: shifted ICU admission and discharge times
- `los`: ICU length of stay
- **Demographics:** `gender`, `insurance`, `marital_status`, `race`
- **Admission timing info**:
    - `anchor_year`: shifted admission year (hospital-level)
    - `anchor_age`: patient age at admission (anchor_year)
    - `anchor_year_group`: actual year range of anchor_year
- **Admission/Discharge info**:
    - `admission_type`: emergency/elective type
    - `admission_location` / `discharge_location`: locations before and after ICU
    - `hospital_expire_flag`: patient died in hospital or not
- **ICU-specific info**:
    - `first_careunit`: first ICU care unit during the stay (usually same as last_careunit)
    - `icu_type`: standardized ICU type
- `assigned_intime` / `assigned_outtime`: ICU times after random year assignment

In [78]:
# icu_patients.nunique()
icu_patients

,subject_id,hadm_id,stay_id,first_careunit,intime,outtime,los,gender,anchor_age,anchor_year,...,admission_location,discharge_location,insurance,language,marital_status,race,hospital_expire_flag,icu_types,assigned_intime,assigned_outtime
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266,F,52,2180,...,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,0,MICU,2015-07-23 14:00:00,2015-07-23 23:50:47
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252,F,86,2150,...,EMERGENCY ROOM,REHAB,Medicare,English,WIDOWED,WHITE,0,MICU,2008-11-02 19:37:00,2008-11-06 17:03:17
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535,F,73,2186,...,EMERGENCY ROOM,HOME HEALTH CARE,Medicare,English,MARRIED,BLACK/AFRICAN AMERICAN,0,MICU,2011-06-27 08:42:00,2011-06-27 20:38:27
3,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032,F,55,2157,...,EMERGENCY ROOM,HOME HEALTH CARE,Private,Other,MARRIED,WHITE,0,SICU,2012-11-20 19:18:02,2012-11-21 22:08:00
4,10001217,27703517,34592300,Surgical Intensive Care Unit (SICU),2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113,F,55,2157,...,PHYSICIAN REFERRAL,HOME HEALTH CARE,Private,Other,MARRIED,WHITE,0,SICU,2012-12-19 15:42:24,2012-12-20 14:27:41
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94453,19999442,26785317,32336619,Surgical Intensive Care Unit (SICU),2148-11-19 14:23:43,2148-11-26 13:12:15,6.950370,M,41,2146,...,PHYSICIAN REFERRAL,REHAB,Medicaid,English,DIVORCED,WHITE,0,SICU,2012-11-19 14:23:43,2012-11-26 13:12:15
94454,19999625,25304202,31070865,Medical/Surgical Intensive Care Unit (MICU/SICU),2139-10-10 19:18:00,2139-10-11 18:21:28,0.960741,M,81,2138,...,EMERGENCY ROOM,HOME HEALTH CARE,Medicare,Modern Greek (1453-),MARRIED,WHITE,0,MICU/SICU,2010-10-10 19:18:00,2010-10-11 18:21:28
94455,19999828,25744818,36075953,Medical Intensive Care Unit (MICU),2149-01-08 18:12:00,2149-01-10 13:11:02,1.790995,F,46,2147,...,TRANSFER FROM HOSPITAL,HOME HEALTH CARE,Medicaid,English,SINGLE,WHITE,0,MICU,2021-01-08 18:12:00,2021-01-10 13:11:02
94456,19999840,21033226,38978960,Surgical Intensive Care Unit (SICU),2164-09-12 09:26:28,2164-09-17 16:35:15,5.297766,M,58,2164,...,EMERGENCY ROOM,DIED,Private,English,WIDOWED,WHITE,1,SICU,2010-09-12 09:26:28,2010-09-17 16:35:15


In [79]:
icu_patients.to_csv(output_folder+'icu_patients.csv', index=False)

In [80]:
icu_patients["assigned_intime"].min(), icu_patients["assigned_intime"].max()

(Timestamp('2008-01-01 08:01:00'), Timestamp('2025-06-23 05:27:15'))

## Dynamics, i.e. Daily Inflow/Outflow in each care unit
- based on the assigned year
- aggregated by day

In [10]:
df = icu_patients.copy()[["subject_id","stay_id", "hadm_id", "assigned_intime", "assigned_outtime", "icu_types"]]

# inflow
inflow_df = df.groupby([df['assigned_intime'].dt.date, 'icu_types']).size().reset_index(name='inflow')
inflow_df.rename(columns={'assigned_intime': 'date'}, inplace=True)

# outflow
outflow_df = df.groupby([df['assigned_outtime'].dt.date, 'icu_types']).size().reset_index(name='outflow')
outflow_df.rename(columns={'assigned_outtime': 'date'}, inplace=True)

merge_df = pd.merge(inflow_df, outflow_df, on=['date', 'icu_types'])
merge_df["outflow"] = merge_df["outflow"].map(lambda x: -x)
merge_df["change"] = merge_df["outflow"] + merge_df["inflow"]

merge_df.to_csv(output_folder+'Dynamics.csv')

In [11]:
# inflow only
inflow_pivot = merge_df.pivot_table(index='date', columns='icu_types', values='inflow', fill_value=0)
inflow_pivot = inflow_pivot.astype("int64")
inflow_pivot["ICU_total_inflow"] = inflow_pivot.sum(axis=1)
# change(inflow-outflow)
merge_pivot = merge_df.pivot_table(index='date', columns='icu_types', values='change', fill_value=0)
merge_pivot = merge_pivot.astype("int64")
merge_pivot["ICU_total_change"] = merge_pivot.sum(axis=1)
# occupancy
occupancy_pivot = merge_pivot.cumsum(axis=0)
occupancy_pivot.drop(columns = ["ICU_total_change"], inplace=True)
occupancy_pivot["ICU_total_occupancy"] = occupancy_pivot.sum(axis=1)

In [12]:
os.makedirs(os.path.dirname(output_folder+'pivot/'), exist_ok=True)

inflow_pivot.to_csv(output_folder+'pivot/inflow_only.csv', index=True)
occupancy_pivot.to_csv(output_folder+ 'pivot/cum_occupancy.csv', index=True)
merge_pivot.to_csv(output_folder+ 'pivot/change(inflow-outflow).csv', index=True)

## Chart Events i.e. clinical events during ICU stay
- from `icu/d_items` and `icu/chartevents`

In [18]:
d_items = pd.read_csv(input_folder+'d_items.csv')
# remove Text data
# d_items = d_items.loc[(d_items["linksto"]=="chartevents")&(d_items["param_type"]!="Text")]
# d_items

### Kept Chart Events
**"General"**
- Daily Weight; Admission Weight (lbs. | Kg.)
- Height | Height (cm)

**"Routine Vital Signs"**
- Heart Rate
- Non Invasive Blood Pressure mean
- Arterial Blood Pressure mean
- Temperature Fahrenheit | Temperature Celsius
- ART BP Mean

**"Respiratory Rate"**
- Respiratory Rate
- 02 saturation pulseoxymetry
- Inspired O2 Fraction

**"Alarm"** - binary
- Heart rate Alarm - High
- Heart rate Alarm - Low
- Resp Alarm - High
- Resp Alarm - Low
- 02 Saturation Pulseoxymetry Alarm - Low
- 02 Saturation Pulseoxymetry Alarm - High

**"Access Lines - Peripheral | Invasive", "IABP", "Dialysis", "Impella", "Cardiovascular"** -> binary

**GCSApacheIIScore**


In [19]:
def chunk_process(chunk, d_items, icu_patients):
    merge_df = pd.merge(chunk, d_items)
    merge_df.drop(columns=['caregiver_id','storetime','warning','abbreviation','linksto', 'unitname', 'lownormalvalue','highnormalvalue'], inplace=True)
    
    # categories which are transformed to binary values
    binary_category = ["Access Lines = Peripheral", "Access Lines - Invasive", "Impella", "Dialysis", "IABP", "Cardiovascular"]
    # labels to keep
    keep_labels = ["Weight", "Height", "Temperature", "Heart Rate", "Non Invasive Blood Pressure mean",
                   "Arterial Blood Pressure mean", "ART BP Mean", "Respiratory Rate", "02 saturation pulseoxymetry",
                   "Inspired O2 Fraction", "Heart rate Alarm - High", "Heart rate Alarm - Low", "Resp Alarm - High",
                   "Resp Alarm - Low",  "02 Saturation Pulseoxymetry Alarm - Low", "02 Saturation Pulseoxymetry Alarm - High",
                   "Low risk (25-50) interventions","High risk (>51) interventions", "Family Meeting", "GcsApacheIIScore"]
    keep_labels += ["if_"+bc for bc in binary_category]

    # Unify units for Selected Features
    # weight
    weight_labels = ["Admission Weight (lbs.)", "Admission Weight (kg)", "Daily Weight"]
    mask = merge_df["label"] == "Admission Weight (lbs.)"
    merge_df.loc[mask, "value"] = merge_df.loc[mask, "value"].astype(float) * 0.453592
    merge_df.loc[merge_df["label"].isin(weight_labels), "valueuom"] = "kg"
    merge_df.loc[merge_df["label"].isin(weight_labels), "label"] = "Weight"
    # height
    height_labels = ["Height", "Height (cm)"]
    mask = merge_df["label"] == "Height"
    merge_df.loc[mask, "value"] = merge_df.loc[mask, "value"].astype(float) * 2.54
    merge_df.loc[merge_df["label"].isin(height_labels), "valueuom"] = "cm"
    merge_df.loc[merge_df["label"].isin(height_labels), "label"] = "Height"
    # temperature
    temperature_labels = ["Temperature Fahrenheit", "Temperature Celsius"]
    mask = merge_df["label"] == "Temperature Fahrenheit"
    merge_df.loc[mask, "value"] = (merge_df.loc[mask, "value"].astype(float) - 32)*5/9
    merge_df.loc[merge_df["label"].isin(temperature_labels), "valueuom"] = "°C"
    merge_df.loc[merge_df["label"].isin(temperature_labels), "label"] = "Temperature"

    # conversion: -> binary
    for bc in binary_category:
        mask = merge_df["category"] == bc
        merge_df.loc[mask, "value"] = 1
        merge_df.loc[mask, "label"] = "if_" + bc
        merge_df.loc[mask, "param_type"] = "Checkbox"

    # drop unnecessary columns, rows and duplicate values
    merge_df.drop(columns=["category", "valuenum"], inplace=True)
    merge_df = merge_df.loc[merge_df["label"].isin(keep_labels)]
    merge_df.drop_duplicates(inplace=True)

    merge_df["charttime"] = pd.to_datetime(merge_df["charttime"])
    merge_df = pd.merge(merge_df, icu_patients[["subject_id", "stay_id", "hadm_id", "intime"]])

    # calculate how long after ICU admission each chart event occurred
    merge_df["stay_days"] = (merge_df["charttime"] - merge_df["intime"]).dt.days
    merge_df["chartdate"] = merge_df["charttime"].dt.date
    # aggregate multiple records per day by mean for numerical
    merge_df["value"] = merge_df["value"].astype(float)
    merge_df = merge_df.groupby(["subject_id", "hadm_id", "stay_id", "chartdate", "intime", "label", "stay_days", "param_type"])["value"].mean().reset_index()

    # for binary values, convert the average back to 0/1
    mask = merge_df["param_type"]=="Checkbox"
    merge_df.loc[mask, "value"] = merge_df.loc[mask, "value"].apply(lambda x: 1 if x>0 else 0)

    return merge_df

### Load, Process and Save
Process the raw `chartevents` in chunks and save them.

In [20]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
data_path = input_folder+'chartevents.csv.gz'
output_path = output_folder + 'selected_chartevents.csv'
chunksize=1000000
# total_rows = sum(1 for _ in open(data_path)) - 1  # ignore header line
total_chunks = 433

In [22]:
chunks = pd.read_csv(data_path, chunksize=chunksize)

In [23]:
print("Processing chunks...")

for i, chunk in enumerate(tqdm(chunks, total=total_chunks, desc="Processing chunks")):
    first_chunk = (i == 0)
    processed_chunk = chunk_process(chunk, d_items, icu_patients)
    processed_chunk.to_csv(output_path, index=False, mode='w' if first_chunk else 'a', header=first_chunk)


Processing chunks...


Processing chunks: 100%|██████████| 433/433 [33:00<00:00,  4.57s/it]    


### Output Description
- `chartdate`: Timestamp when the charted event happened
- `itemid`: Unique identifier for the charted item
- `value`: Measured or recorded value of the item
- `label`: Human-readable name of the item (after unit unification and mapping)
- `param_type`: Type of the value (e.g., Numeric, Checkbox)
- `intime`
- `stay_days`: Number of days since ICU admission (calculated as `charttime` - `intime`)
    - note that there might be issues of erroneous records or smth corresponding to admission checks; handle these during preprocessing.
- `valueuom`: unit of value

In [21]:
processed_chunk

,subject_id,hadm_id,stay_id,chartdate,intime,label,stay_days,param_type,value
0,19979220,27248006,39947778,2163-12-20,2163-12-20 18:35:25,Heart Rate,0,Numeric,70.000000
1,19979220,27248006,39947778,2163-12-20,2163-12-20 18:35:25,Height,-1,Numeric,180.170000
2,19979220,27248006,39947778,2163-12-20,2163-12-20 18:35:25,High risk (>51) interventions,0,Checkbox,1.000000
3,19979220,27248006,39947778,2163-12-20,2163-12-20 18:35:25,Non Invasive Blood Pressure mean,0,Numeric,77.500000
4,19979220,27248006,39947778,2163-12-20,2163-12-20 18:35:25,Respiratory Rate,0,Numeric,18.750000
...,...,...,...,...,...,...,...,...,...
14607,19999987,23865745,36195440,2145-11-04,2145-11-02 22:59:00,Resp Alarm - High,1,Numeric,35.000000
14608,19999987,23865745,36195440,2145-11-04,2145-11-02 22:59:00,Resp Alarm - Low,1,Numeric,8.000000
14609,19999987,23865745,36195440,2145-11-04,2145-11-02 22:59:00,Respiratory Rate,1,Numeric,23.578947
14610,19999987,23865745,36195440,2145-11-04,2145-11-02 22:59:00,Temperature,1,Numeric,37.479167


In [24]:
selected_chartevents = pd.read_csv(output_folder+'selected_chartevents_Nov5.csv')
selected_chartevents

,subject_id,hadm_id,stay_id,chartdate,intime,label,stay_days,param_type,value
0,10000032,29079034,39553978,2180-07-23,2180-07-23 14:00:00,Heart Rate,0,Numeric,96.500000
1,10000032,29079034,39553978,2180-07-23,2180-07-23 14:00:00,Heart rate Alarm - High,0,Numeric,120.000000
2,10000032,29079034,39553978,2180-07-23,2180-07-23 14:00:00,Height,-1,Numeric,152.200000
3,10000032,29079034,39553978,2180-07-23,2180-07-23 14:00:00,High risk (>51) interventions,0,Checkbox,1.000000
4,10000032,29079034,39553978,2180-07-23,2180-07-23 14:00:00,Non Invasive Blood Pressure mean,0,Numeric,62.300000
...,...,...,...,...,...,...,...,...,...
6249948,19999987,23865745,36195440,2145-11-04,2145-11-02 22:59:00,Resp Alarm - High,1,Numeric,35.000000
6249949,19999987,23865745,36195440,2145-11-04,2145-11-02 22:59:00,Resp Alarm - Low,1,Numeric,8.000000
6249950,19999987,23865745,36195440,2145-11-04,2145-11-02 22:59:00,Respiratory Rate,1,Numeric,23.578947
6249951,19999987,23865745,36195440,2145-11-04,2145-11-02 22:59:00,Temperature,1,Numeric,37.479167


In [24]:
icu_patients

,subject_id,hadm_id,stay_id,first_careunit,intime,outtime,los,gender,anchor_age,anchor_year,...,admission_location,discharge_location,insurance,language,marital_status,race,hospital_expire_flag,icu_types,assigned_intime,assigned_outtime
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266,F,52,2180,...,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,0,MICU,2023-07-23 14:00:00,2023-07-23 23:50:47
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252,F,86,2150,...,EMERGENCY ROOM,REHAB,Medicare,English,WIDOWED,WHITE,0,MICU,2023-11-02 19:37:00,2023-11-06 17:03:17
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535,F,73,2186,...,EMERGENCY ROOM,HOME HEALTH CARE,Medicare,English,MARRIED,BLACK/AFRICAN AMERICAN,0,MICU,2023-06-27 08:42:00,2023-06-27 20:38:27
3,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032,F,55,2157,...,EMERGENCY ROOM,HOME HEALTH CARE,Private,Other,MARRIED,WHITE,0,SICU,2023-11-20 19:18:02,2023-11-21 22:08:00
4,10001217,27703517,34592300,Surgical Intensive Care Unit (SICU),2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113,F,55,2157,...,PHYSICIAN REFERRAL,HOME HEALTH CARE,Private,Other,MARRIED,WHITE,0,SICU,2023-12-19 15:42:24,2023-12-20 14:27:41
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94453,19999442,26785317,32336619,Surgical Intensive Care Unit (SICU),2148-11-19 14:23:43,2148-11-26 13:12:15,6.950370,M,41,2146,...,PHYSICIAN REFERRAL,REHAB,Medicaid,English,DIVORCED,WHITE,0,SICU,2023-11-19 14:23:43,2023-11-26 13:12:15
94454,19999625,25304202,31070865,Medical/Surgical Intensive Care Unit (MICU/SICU),2139-10-10 19:18:00,2139-10-11 18:21:28,0.960741,M,81,2138,...,EMERGENCY ROOM,HOME HEALTH CARE,Medicare,Modern Greek (1453-),MARRIED,WHITE,0,MICU/SICU,2023-10-10 19:18:00,2023-10-11 18:21:28
94455,19999828,25744818,36075953,Medical Intensive Care Unit (MICU),2149-01-08 18:12:00,2149-01-10 13:11:02,1.790995,F,46,2147,...,TRANSFER FROM HOSPITAL,HOME HEALTH CARE,Medicaid,English,SINGLE,WHITE,0,MICU,2023-01-08 18:12:00,2023-01-10 13:11:02
94456,19999840,21033226,38978960,Surgical Intensive Care Unit (SICU),2164-09-12 09:26:28,2164-09-17 16:35:15,5.297766,M,58,2164,...,EMERGENCY ROOM,DIED,Private,English,WIDOWED,WHITE,1,SICU,2023-09-12 09:26:28,2023-09-17 16:35:15


In [ ]:
df = pd.read_csv("Merged_Dataset/results.csv")

,subject_id,hadm_id,stay_id,assigned_intime,assigned_outtime,chartdate,actual_los,predicted_los,abs_error
0,17057610,26086792,30045625,2020-05-17 22:24:28,2020-06-16 18:16:36,2200-06-08,8,14.580319,6.580319
1,17057610,26086792,30045625,2020-05-17 22:24:28,2020-06-16 18:16:36,2200-06-08,8,14.182450,6.182450
2,17057610,26086792,30045625,2020-05-17 22:24:28,2020-06-16 18:16:36,2200-06-08,8,14.969455,6.969455
3,17057610,26086792,30045625,2020-05-17 22:24:28,2020-06-16 18:16:36,2200-06-08,8,9.247801,1.247801
4,17057610,26086792,30045625,2020-05-17 22:24:28,2020-06-16 18:16:36,2200-06-08,8,9.103970,1.103970
...,...,...,...,...,...,...,...,...,...
242198,17598702,20473158,37693858,2025-06-23 05:27:15,2025-06-27 15:00:40,2144-06-25,2,1.189700,0.810300
242199,17598702,20473158,37693858,2025-06-23 05:27:15,2025-06-27 15:00:40,2144-06-25,2,1.001645,0.998355
242200,17598702,20473158,37693858,2025-06-23 05:27:15,2025-06-27 15:00:40,2144-06-24,3,1.001645,1.998355
242201,17598702,20473158,37693858,2025-06-23 05:27:15,2025-06-27 15:00:40,2144-06-24,3,1.001645,1.998355


In [42]:
df["assigned_intime"] = pd.to_datetime(df["assigned_intime"])
df["date"] = df["assigned_intime"].dt.date
print(df["date"].min())
print(df["date"].max())

2008-01-01
2025-06-23
